# Variance Inflation Factor — Detecting Multicollinearity

A correlation matrix is the usual first check for redundant predictors, and it has a blind spot: it
only sees **pairs**. A feature that is a linear combination of nine others can be weakly correlated
with every single one of them individually, and still carry no information the others don't already
have. The correlation matrix will show nothing wrong.

The **Variance Inflation Factor** finds exactly that case, by asking of each predictor: how well do
*all the others together* explain it?

## Learning objectives

- Explain why a pairwise correlation matrix cannot detect multi-way collinearity
- Compute VIF for every predictor with `statsmodels`
- Interpret VIF values against the usual rules of thumb
- Show that a feature built from many others has a high VIF despite low pairwise correlations

## Background

This notebook assumes linear regression and the $R^2$ score, and the idea that a regression
coefficient has a standard error attached to it. Everything specific to VIF — the auxiliary
regression and the inflation formula — is developed in section 3 where it is computed.

**Prerequisites:** `U1-3_Regression-1_ModelCompare` ($R^2$, linear regression)

**Dataset:** none — ten synthetic predictors generated in section 1. Three constructions of the
tenth predictor are provided and selected by toggling comments.

**References:** https://www.statsmodels.org/stable/generated/statsmodels.stats.outliers_influence.variance_inflation_factor.html

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)
pd.set_option("display.precision", 4)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers

## 1. Create the data

Nine independent standard-normal predictors, plus a tenth built three different ways. The whole
point of the notebook is the contrast between them, so **all three are worth running**:

| Option | How `x10` is built | What you should see |
|---|---|---|
| Independent | `np.random.randn(n)` — unrelated to the rest | Every VIF near 1. This is the control case. |
| Fully determined | $x_{10} = 0.2x_1 + 0.2x_2 - 0.2x_3 + \cdots$ with **no** noise term | An exact linear dependency. VIF explodes toward infinity for `x10` *and* for every feature used to build it. |
| Partially determined (active) | the same combination plus `0.1*np.random.randn(n)` | The realistic middle case: elevated VIF, but finite. |

Notice what the middle and bottom options have in common — each coefficient is small (0.05 to 0.3),
so `x10` is only *weakly* correlated with any individual $x_j$. The redundancy is spread across all
nine. That is precisely the situation section 2 will fail to detect and section 3 will catch.

In [2]:
import numpy as np
import pandas as pd

n = 1000

# Independent random variables
x1 = np.random.randn(n)
x2 = np.random.randn(n)
x3 = np.random.randn(n)
x4 = np.random.randn(n)
x5 = np.random.randn(n)
x6 = np.random.randn(n)
x7 = np.random.randn(n)
x8 = np.random.randn(n)
x9 = np.random.randn(n)

# Create a 10th variable as a linear combination of all others
x10 = np.random.randn(n)                                   # Uncorrelated with others
#x10 = (0.2*x1 + 0.2*x2 - 0.2*x3 + 0.2*x4 + 0.2*x5 
       #- 0.2*x6 + 0.2*x7 + 0.2*x8 - 0.2*x9 
       #+ 0.0*np.random.randn(n))                          # Correlated with all
x10 = (0.2*x1 + 0.1*x2 - 0.3*x3 + 0.25*x4 + 0.15*x5 
       - 0.2*x6 + 0.1*x7 + 0.05*x8 - 0.1*x9 + 0.1*np.random.randn(n))              # Perfectly collinear

## 2. What the correlation matrix shows

The standard first look. Each entry is a **pairwise** correlation,

$$\rho_{jk} = \frac{\operatorname{Cov}(X_j, X_k)}{\sigma_{X_j}\sigma_{X_k}}$$

and with the active option every one of them is small — `x10`'s largest correlation with any single
predictor is around 0.3, which few people would flag as a problem.

That verdict is wrong, and the next section shows why. Correlation asks "does $X_j$ track $X_k$?"
one pair at a time; it has no way to ask "do the other nine *together* pin down $X_{10}$?"

In [3]:
# Put in DataFrame
df = pd.DataFrame({
    "x1": x1,
    "x2": x2,
    "x3": x3,
    "x4": x4,
    "x5": x5,
    "x6": x6,
    "x7": x7,
    "x8": x8,
    "x9": x9,
    "x10": x10
})

df.corr()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10
x1,1.0000,-0.0279,0.0461,0.0241,-0.0108,-0.0058,0.0239,-0.0403,-0.0978,0.3636
x2,-0.0279,1.0000,-0.0155,0.0446,0.0274,-0.0347,0.0258,-0.0078,-0.0042,0.2415
x3,0.0461,-0.0155,1.0000,0.0098,0.0086,-0.0104,-0.0205,0.0153,-0.0152,-0.5229
x4,0.0241,0.0446,0.0098,1.0000,-0.0417,-0.0385,-0.0133,0.0007,-0.0288,0.4623
x5,-0.0108,0.0274,0.0086,-0.0417,1.0000,0.0016,-0.0129,-0.0295,0.0507,0.2302
x6,-0.0058,-0.0347,-0.0104,-0.0385,0.0016,1.0000,-0.0258,0.0062,0.0241,-0.3895
x7,0.0239,0.0258,-0.0205,-0.0133,-0.0129,-0.0258,1.0000,-0.0151,-0.0574,0.2200
x8,-0.0403,-0.0078,0.0153,0.0007,-0.0295,0.0062,-0.0151,1.0000,0.0148,0.0444
x9,-0.0978,-0.0042,-0.0152,-0.0288,0.0507,0.0241,-0.0574,0.0148,1.0000,-0.2444
x10,0.3636,0.2415,-0.5229,0.4623,0.2302,-0.3895,0.2200,0.0444,-0.2444,1.0000


## 3. Compute VIF

VIF answers the question correlation cannot. For each predictor $X_j$, fit an **auxiliary
regression** using every *other* predictor as the input:

$$\hat{X}_j = \sum_{\substack{k=1 \\ k \neq j}}^{p} \beta_k X_k$$

Let $R_j^2$ be that regression's coefficient of determination — how much of $X_j$ the others
explain. The variance inflation factor is

$$\text{VIF}(X_j) = \frac{1}{1 - R_j^2}$$

The name is literal. In a linear regression, the variance of the estimated coefficient
$\hat{\beta}_j$ is proportional to $\text{VIF}(X_j)$, so this is the factor by which
collinearity inflates that coefficient's standard error. A redundant predictor doesn't bias the
model; it makes the coefficient estimates **unstable** — large, opposite-signed, and liable to
swing wildly on a resample.

Reading the numbers:

| $R_j^2$ | VIF | Reading |
|---|---|---|
| 0 | 1 | Completely independent of the others |
| 0.5 | 2 | Mild, generally ignorable |
| 0.8 | 5 | Worth attention |
| 0.9 | 10 | The common "drop it" threshold |
| $\to 1$ | $\to \infty$ | Exact linear dependency |

Compare the output below against the correlation matrix above. `x10` should stand out here despite
being unremarkable there — and so should the features that went into building it, since the
dependency is symmetric.

In [4]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Compute VIFs without adding a constant
vif = pd.Series(
    [variance_inflation_factor(df.values, i) 
     for i in range(df.shape[1])],
    index=df.columns
)

print("Variance Inflation Factors (no constant):")
print(vif)


Variance Inflation Factors (no constant):
x1      5.0298
x2      2.2070
x3     10.0872
x4      6.9780
x5      3.1742
x6      4.9425
x7      2.0409
x8      1.2197
x9      2.2200
x10    30.2976
dtype: float64


## 4. Review

- **Correlation is pairwise; collinearity need not be.** A predictor can be a linear combination of
  many others while correlating weakly with each one individually.
- **VIF regresses each predictor on all the others**, and reports
  $\text{VIF}(X_j) = 1/(1 - R_j^2)$.
- **The name is literal**: VIF is the factor by which collinearity inflates the variance of the
  estimated coefficient $\hat{\beta}_j$. Collinearity makes coefficients unstable, not biased.
- **Rules of thumb**: VIF near 1 is clean, above 5 deserves attention, above 10 is usually cause to
  drop or combine features. An exact dependency sends VIF to infinity.
- **The dependency is symmetric.** A high VIF implicates a *group* of features, not one culprit —
  which is why the fix is to drop or combine, guided by what the features mean.